**Connect Scripts**

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from Scripts import FileHandler as fh

**Inspect Data Set**

In [ ]:
# load data set
import kagglehub
from pathlib import Path

downloadPath = kagglehub.dataset_download('yasserh/titanic-dataset')
dataPath = Path(downloadPath)

print(f'Content of {dataPath}:')
for item in dataPath.iterdir():
    print(f"    -{item.name} ({'Folder' if item.is_dir() else 'File'})")


**Inspect Data Set**

In [ ]:
# Inspect data set
import pandas as pd

dfTitanic = pd.read_csv(f'{downloadPath}/Titanic-Dataset.csv')

# print first 5 lines
display(dfTitanic.head())

# check data set shape
print(f"\n Titanic Dataset shape : {dfTitanic.shape}")

# check available data types
print("\n Data Type Count")
print(dfTitanic.dtypes.value_counts())

# check missing values in data set
print("\n Missing values in Data set")
dfTitanic.info()

# statistical summary
print("\n statistical summary")
display(dfTitanic.describe())

# check distribution
print("\n Data set distribution")
print(dfTitanic['Survived'].value_counts())
print(dfTitanic['Survived'].value_counts(normalize=True)* 100)


**Train/Validate/Test Split**
- avoid cross contamination (don't need to get influence by Training data)
- 70% : Train ; 15% : Validation ; 15% : Test

In [ ]:
from sklearn.model_selection import train_test_split

dfTrain, dfTemp = train_test_split(dfTitanic, test_size=0.30, random_state=42)
dfVal, dfTest = train_test_split(dfTemp, test_size=0.5, random_state=42)

print(f"Dataset Size: {len(dfTitanic)} | Train Size: {len(dfTrain)} | Validate Size: {len(dfVal)} | Test Size: {len(dfTest)}")
fh.saveDataSets(dfTrain, dfVal, dfTest, "titanicdataset/preprocessing", "01_datasplit")

**Handle Missing Values**
- Drop the columns
- Imputation/ Replace
    - mean
    - meadian
    - frequent
    - null value

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("titanicdataset/preprocessing", "01_datasplit")

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
# drop columns
dropList = ['Cabin']

print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfVal.drop(columns=dropList, errors='ignore')
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
# replace with median value
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
medianReplaceList = ['Age']
print("missing in Train: ", dfTrain['Age'].isnull().sum())
print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
for column in medianReplaceList:
    if column in dfTrain.columns:
        imputer.fit(dfTrain[[column]])
        print(f"Imputed(median) value: {imputer.statistics_}")

        dfTrain[[column]] = imputer.transform(dfTrain[[column]])

        if column in dfVal:
            dfVal[[column]] = imputer.transform(dfVal[[column]])

        if column in dfTest:
            dfTest[[column]] = imputer.transform(dfTest[[column]])
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
print("missing in Train: ", dfTrain['Age'].isnull().sum())

In [ ]:
# replace with frequent value

frequentReplaceList = ['Embarked']

print("missing in Train: ", dfTrain['Embarked'].isnull().sum())
for column in frequentReplaceList:
    if column in dfTrain.columns:
        modeValue = dfTrain[column].mode()[0]
        print(f"Imputed(frequent) value: {modeValue}")

        dfTrain[column] = dfTrain[column].fillna(modeValue)

        if column in dfVal:
            dfVal[column] = dfVal[column].fillna(modeValue)

        if column in dfTest:
            dfTest[column] = dfTest[column].fillna(modeValue)

print("missing in Train: ", dfTrain['Embarked'].isnull().sum())

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "titanicdataset/preprocessing", "02_handlemissingvalues")